# Demographic Parity & Fairness Balancing

In [1]:
INPUT_CSV         = "/home/tariq/Desktop/Data-balace/adult_census_clean.csv"
SENSITIVE_COL     = "sex"
GROUP_A_VALUE     = "Male"
GROUP_B_VALUE     = "Female"
TARGET_COL        = "income"
POSITIVE_LABEL    = ">50K"
FAVORABLE_LABEL   = 1         
# Set FAVORABLE_LABEL = 1
# Set FAVORABLE_LABEL = 0 
EPSILON           = 0.0001
MAX_ITERATIONS    = 100_000
RANDOM_SEED       = 42
OUTPUT_BEFORE_CSV = "parity_before.csv"
OUTPUT_AFTER_CSV  = "parity_after.csv"
OUTPUT_BALANCED   = "balanced_dataset.csv"


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

np.random.seed(RANDOM_SEED)
print("✅ Imports successful.")


✅ Imports successful.


In [4]:
df = pd.read_csv(INPUT_CSV)

# Strip whitespace from string columns (common in UCI Adult dataset)
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSensitive column '{SENSITIVE_COL}' value counts:")
print(df[SENSITIVE_COL].value_counts())
print(f"\nTarget column '{TARGET_COL}' value counts:")
print(df[TARGET_COL].value_counts())
# df.head()


Dataset loaded: 30,162 rows × 15 columns

Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']

Sensitive column 'sex' value counts:
sex
Male      20380
Female     9782
Name: count, dtype: int64

Target column 'income' value counts:
income
<=50K    22654
>50K      7508
Name: count, dtype: int64


## Encode Target Column as Binary

In [5]:
BINARY_COL = "_label_binary"
df[BINARY_COL] = (df[TARGET_COL] == POSITIVE_LABEL).astype(int)

print(f"Binary label distribution (1 = '{POSITIVE_LABEL}'):")
print(df[BINARY_COL].value_counts())


Binary label distribution (1 = '>50K'):
_label_binary
0    22654
1     7508
Name: count, dtype: int64


## Demographic Parity Function

In [6]:
def compute_demographic_parity(data, sensitive_col, binary_col, group_a, group_b, favorable_label=1):
    """
    Computes demographic parity metrics for two groups.

    Parameters
    ----------
    data           : pd.DataFrame
    sensitive_col  : str   — column with group labels
    binary_col     : str   — column with 0/1 labels
    group_a        : str   — e.g. 'Male'
    group_b        : str   — e.g. 'Female'
    favorable_label: int   — 1 or 0, which outcome to measure parity on

    Returns
    -------
    dict with counts, rates, difference, ratio
    """
    mask_a = data[sensitive_col] == group_a
    mask_b = data[sensitive_col] == group_b

    total_a  = mask_a.sum()
    total_b  = mask_b.sum()

    fav_a    = ((data[binary_col] == favorable_label) & mask_a).sum()
    fav_b    = ((data[binary_col] == favorable_label) & mask_b).sum()

    rate_a   = fav_a / total_a if total_a > 0 else 0.0
    rate_b   = fav_b / total_b if total_b > 0 else 0.0

    dp_diff  = abs(rate_a - rate_b)                        # Should be < epsilon when balanced
    dp_ratio = (min(rate_a, rate_b) / max(rate_a, rate_b)
                if max(rate_a, rate_b) > 0 else 1.0)       # 1.0 = perfect parity

    return {
        f"Total_{group_a}"                    : int(total_a),
        f"Total_{group_b}"                    : int(total_b),
        f"{group_a}_favorable_count"          : int(fav_a),
        f"{group_b}_favorable_count"          : int(fav_b),
        f"{group_a}_positive_rate"            : round(rate_a, 6),
        f"{group_b}_positive_rate"            : round(rate_b, 6),
        "Demographic_Parity_Difference"       : round(dp_diff, 6),
        "Demographic_Parity_Ratio"            : round(dp_ratio, 6),
        "Favorable_Label"                     : favorable_label,
        "Is_Fair (diff < epsilon)"            : dp_diff < EPSILON,
    }

print("✅ Function defined.")


✅ Function defined.


## 📋 Compute & Save Parity — BEFORE Balancing

In [7]:
parity_before = compute_demographic_parity(
    df, SENSITIVE_COL, BINARY_COL,
    GROUP_A_VALUE, GROUP_B_VALUE,
    favorable_label=FAVORABLE_LABEL
)

parity_before_df = pd.DataFrame([parity_before])
parity_before_df.to_csv(OUTPUT_BEFORE_CSV, index=False)

print("=" * 55)
print("  DEMOGRAPHIC PARITY — BEFORE BALANCING")
print("=" * 55)
for k, v in parity_before.items():
    print(f"  {k:<42} {v}")
print(f"\n💾 Saved to: {OUTPUT_BEFORE_CSV}")


  DEMOGRAPHIC PARITY — BEFORE BALANCING
  Total_Male                                 20380
  Total_Female                               9782
  Male_favorable_count                       6396
  Female_favorable_count                     1112
  Male_positive_rate                         0.313837
  Female_positive_rate                       0.113678
  Demographic_Parity_Difference              0.200159
  Demographic_Parity_Ratio                   0.36222
  Favorable_Label                            1
  Is_Fair (diff < epsilon)                   False

💾 Saved to: parity_before.csv


## ⚖️ Fairness Balancing Algorithm

In [20]:
def balance_dataset(data, sensitive_col, binary_col,
                    group_a, group_b, favorable_label,
                    positive_label, epsilon, max_iterations, seed):
    """
    Iteratively removes rows to equalize positive rates between group_a and group_b.
    """
    rng = np.random.default_rng(seed)
    balanced = data.copy()
    log = []
    unfav_label = 1 - favorable_label
    negative_label = "<=50K" if positive_label == ">50K" else f"NOT {positive_label}"

    for iteration in range(1, max_iterations + 1):

        mask_a = balanced[sensitive_col] == group_a
        mask_b = balanced[sensitive_col] == group_b

        X  = mask_a.sum()
        Y  = mask_b.sum()
        X1 = ((balanced[binary_col] == favorable_label) & mask_a).sum()
        Y1 = ((balanced[binary_col] == favorable_label) & mask_b).sum()
        X0 = X - X1
        Y0 = Y - Y1

        if X == 0 or Y == 0:
            print(f"\n⚠️  Stopping at iteration {iteration}: one group has been fully removed.")
            break

        rate_a = X1 / X
        rate_b = Y1 / Y
        diff   = abs(rate_a - rate_b)

        log.append({
            "iteration": iteration,
            "X": X, "Y": Y, "X1": X1, "Y1": Y1,
            "rate_a": round(rate_a, 6), "rate_b": round(rate_b, 6),
            "diff": round(diff, 6),
        })

        # Convergence check
        if diff <= epsilon:
            print(f"\n{'='*80}")
            print(f"  ✅ CONVERGED at iteration {iteration}")
            print(f"{'='*80}")
            _print_state(iteration, group_a, group_b, X, Y, X1, Y1, X0, Y0,
                         rate_a, rate_b, diff, "CONVERGED", positive_label, negative_label)
            break

        case_applied = ""
        removed = False

        # ── Case 1: X > Y and rate_a < rate_b ────────────────────────────
        if X > Y and rate_a < rate_b:
            case_applied = f"Case 1: {group_a} is bigger & rate is lower → remove {group_a} with {negative_label}"
            candidates = balanced.index[mask_a & (balanced[binary_col] == unfav_label)]
            if len(candidates) > 0:
                drop_idx = rng.choice(candidates)
                balanced = balanced.drop(index=drop_idx)
                removed = True

        # ── Case 2: X > Y and rate_a > rate_b ────────────────────────────
        elif X > Y and rate_a > rate_b:
            case_applied = f"Case 2: {group_a} is bigger & rate is higher → remove {group_a} with {positive_label}"
            candidates = balanced.index[mask_a & (balanced[binary_col] == favorable_label)]
            if len(candidates) > 0:
                drop_idx = rng.choice(candidates)
                balanced = balanced.drop(index=drop_idx)
                removed = True

        # ── Case 3: Y > X and rate_a > rate_b ────────────────────────────
        elif Y > X and rate_a > rate_b:
            case_applied = f"Case 3: {group_b} is bigger & {group_a} rate is higher → remove {group_b} with {negative_label}"
            candidates = balanced.index[mask_b & (balanced[binary_col] == unfav_label)]
            if len(candidates) > 0:
                drop_idx = rng.choice(candidates)
                balanced = balanced.drop(index=drop_idx)
                removed = True

        # ── Case 4: Y > X and rate_a < rate_b ────────────────────────────
        elif Y > X and rate_a < rate_b:
            case_applied = f"Case 4: {group_b} is bigger & {group_b} rate is higher → remove {group_b} with {positive_label}"
            candidates = balanced.index[mask_b & (balanced[binary_col] == favorable_label)]
            if len(candidates) > 0:
                drop_idx = rng.choice(candidates)
                balanced = balanced.drop(index=drop_idx)
                removed = True

        # ── Tie-breaker: X == Y ───────────────────────────────────────────
        else:
            if rate_a >= rate_b:
                case_applied = f"Tie: X==Y, {group_a} rate higher → remove {group_a} with {positive_label}"
                candidates = balanced.index[mask_a & (balanced[binary_col] == favorable_label)]
            else:
                case_applied = f"Tie: X==Y, {group_b} rate higher → remove {group_b} with {positive_label}"
                candidates = balanced.index[mask_b & (balanced[binary_col] == favorable_label)]
            if len(candidates) > 0:
                drop_idx = rng.choice(candidates)
                balanced = balanced.drop(index=drop_idx)
                removed = True

        if not removed:
            print(f"\n⚠️  Stopping at iteration {iteration}: no valid candidates to remove.")
            break

        if iteration <= 10 or iteration % 500 == 0 or iteration % 5000 == 0:
            _print_state(iteration, group_a, group_b, X, Y, X1, Y1, X0, Y0,
                         rate_a, rate_b, diff, case_applied, positive_label, negative_label)

    else:
        print(f"\n⚠️  Reached MAX_ITERATIONS ({max_iterations:,}) without full convergence.")

    return balanced, log


def _print_state(iteration, group_a, group_b, X, Y, X1, Y1, X0, Y0,
                 rate_a, rate_b, diff, action, positive_label, negative_label):
    """Print detailed state for one iteration."""
    print(f"\n─── Iteration {iteration:,} ─────────────────────────────────────────────────")
    print(f"  {group_a:<8} (X)  = {X:>7,}   {group_a:<3} {positive_label} (Fav) (X1) = {X1:>6,}    {group_a:<3} {negative_label} (X0) = {X0:>6,}   {group_a:<3} {positive_label} (Fav) (X1) rate = {X1}/{X} = {rate_a:.6f}")
    print(f"  {group_b:<8} (Y)  = {Y:>7,}   {group_b:<3} {positive_label} (Fav) (Y1) = {Y1:>6,}    {group_b:<3} {negative_label} (Y0) = {Y0:>6,}   {group_b:<3} {positive_label} (Fav) (Y1) rate = {Y1}/{Y} = {rate_b:.6f}")
    print(f"  DP diff = |{rate_a:.6f} - {rate_b:.6f}| = {diff:.6f}")
    print(f"  Action  → {action}")


print("✅ Balancing function defined.")


✅ Balancing function defined.


## 🚀 Run the Balancing Algorithm

In [21]:
print("Starting balancing algorithm...\n")

balanced_df, iteration_log = balance_dataset(
    df,
    sensitive_col   = SENSITIVE_COL,
    binary_col      = BINARY_COL,
    group_a         = GROUP_A_VALUE,
    group_b         = GROUP_B_VALUE,
    favorable_label = FAVORABLE_LABEL,
    positive_label  = POSITIVE_LABEL,
    epsilon         = EPSILON,
    max_iterations  = MAX_ITERATIONS,
    seed            = RANDOM_SEED,
)

rows_removed = len(df) - len(balanced_df)
print(f"\nOriginal dataset : {len(df):,} rows")
print(f"Balanced dataset : {len(balanced_df):,} rows")
print(f"Rows removed     : {rows_removed:,} ({rows_removed/len(df)*100:.1f}%)")


Starting balancing algorithm...


─── Iteration 1 ─────────────────────────────────────────────────
  Male     (X)  =  20,380   Male >50K (Fav) (X1) =  6,396    Male <=50K (X0) = 13,984   Male >50K (Fav) (X1) rate = 6396/20380 = 0.313837
  Female   (Y)  =   9,782   Female >50K (Fav) (Y1) =  1,112    Female <=50K (Y0) =  8,670   Female >50K (Fav) (Y1) rate = 1112/9782 = 0.113678
  DP diff = |0.313837 - 0.113678| = 0.200159
  Action  → Case 2: Male is bigger & rate is higher → remove Male with >50K

─── Iteration 2 ─────────────────────────────────────────────────
  Male     (X)  =  20,379   Male >50K (Fav) (X1) =  6,395    Male <=50K (X0) = 13,984   Male >50K (Fav) (X1) rate = 6395/20379 = 0.313803
  Female   (Y)  =   9,782   Female >50K (Fav) (Y1) =  1,112    Female <=50K (Y0) =  8,670   Female >50K (Fav) (Y1) rate = 1112/9782 = 0.113678
  DP diff = |0.313803 - 0.113678| = 0.200125
  Action  → Case 2: Male is bigger & rate is higher → remove Male with >50K

─── Iteration 3 ────────────

## 💾 Save Balanced Dataset

In [22]:
# Drop the helper binary column before saving
balanced_df.drop(columns=[BINARY_COL]).to_csv(OUTPUT_BALANCED, index=False)
print(f"✅ Balanced dataset saved to: {OUTPUT_BALANCED}")


✅ Balanced dataset saved to: balanced_dataset.csv


## 📋 Compute & Save Parity — AFTER Balancing

In [23]:
parity_after = compute_demographic_parity(
    balanced_df, SENSITIVE_COL, BINARY_COL,
    GROUP_A_VALUE, GROUP_B_VALUE,
    favorable_label=FAVORABLE_LABEL
)

parity_after_df = pd.DataFrame([parity_after])
parity_after_df.to_csv(OUTPUT_AFTER_CSV, index=False)

print("=" * 55)
print("  DEMOGRAPHIC PARITY — AFTER BALANCING")
print("=" * 55)
for k, v in parity_after.items():
    print(f"  {k:<42} {v}")
print(f"\n💾 Saved to: {OUTPUT_AFTER_CSV}")


  DEMOGRAPHIC PARITY — AFTER BALANCING
  Total_Male                                 15779
  Total_Female                               9782
  Male_favorable_count                       1795
  Female_favorable_count                     1112
  Male_positive_rate                         0.113759
  Female_positive_rate                       0.113678
  Demographic_Parity_Difference              8.1e-05
  Demographic_Parity_Ratio                   0.999291
  Favorable_Label                            1
  Is_Fair (diff < epsilon)                   True

💾 Saved to: parity_after.csv


## 🔍 Side-by-Side Comparison

In [24]:
comparison = pd.DataFrame({
    "Metric"  : list(parity_before.keys()),
    "Before"  : list(parity_before.values()),
    "After"   : list(parity_after.values()),
})

print(comparison.to_string(index=False))


                       Metric    Before     After
                   Total_Male     20380     15779
                 Total_Female      9782      9782
         Male_favorable_count      6396      1795
       Female_favorable_count      1112      1112
           Male_positive_rate  0.313837  0.113759
         Female_positive_rate  0.113678  0.113678
Demographic_Parity_Difference  0.200159  0.000081
     Demographic_Parity_Ratio   0.36222  0.999291
              Favorable_Label         1         1
     Is_Fair (diff < epsilon)     False      True


## 📝 Summary Report

In [42]:
print("=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"  Input file              : {INPUT_CSV}")
print(f"  Sensitive column        : {SENSITIVE_COL}")
print(f"  Group A                 : {GROUP_A_VALUE}")
print(f"  Group B                 : {GROUP_B_VALUE}")
print(f"  Target column           : {TARGET_COL}")
print(f"  Favorable label         : {FAVORABLE_LABEL} ('{POSITIVE_LABEL}')")
print(f"  Epsilon (ε)             : {EPSILON}")
print()
print(f"  Rows before balancing   : {len(df):,}")
print(f"  Rows after balancing    : {len(balanced_df):,}")
print(f"  Rows removed            : {rows_removed:,}")
print(f"  Iterations run          : {len(iteration_log):,}")
print()
print(f"  Parity diff BEFORE      : {parity_before['Demographic_Parity_Difference']:.4f}")
print(f"  Parity diff AFTER       : {parity_after['Demographic_Parity_Difference']:.4f}")
print(f"  Parity ratio BEFORE     : {parity_before['Demographic_Parity_Ratio']:.4f}")
print(f"  Parity ratio AFTER      : {parity_after['Demographic_Parity_Ratio']:.4f}")
print()
print(f"  Fair after balancing?   : {parity_after['Is_Fair (diff < epsilon)']}")
print("=" * 60)
print()
print("  Output files:")
print(f"    {OUTPUT_BEFORE_CSV}  — Parity metrics before")
print(f"    {OUTPUT_AFTER_CSV}   — Parity metrics after")
print(f"    {OUTPUT_BALANCED}   — Balanced dataset")
print(f"    demographic_parity_plot.png")
print(f"    group_counts_plot.png")


  FINAL SUMMARY
  Input file              : /home/tariq/Desktop/Data-balace/adult_census_clean.csv
  Sensitive column        : sex
  Group A                 : Male
  Group B                 : Female
  Target column           : income
  Favorable label         : 1 ('>50K')
  Epsilon (ε)             : 0.0001

  Rows before balancing   : 30,162
  Rows after balancing    : 25,561
  Rows removed            : 4,601
  Iterations run          : 4,602

  Parity diff BEFORE      : 0.2002
  Parity diff AFTER       : 0.0001
  Parity ratio BEFORE     : 0.3622
  Parity ratio AFTER      : 0.9993

  Fair after balancing?   : True

  Output files:
    parity_before.csv  — Parity metrics before
    parity_after.csv   — Parity metrics after
    balanced_dataset.csv   — Balanced dataset
    demographic_parity_plot.png
    group_counts_plot.png
